# 15. Alert lead time vs. labeled stress periods (Week 6)

**Owner:** Rishi. Task from the plan: *measure alert lead time vs. labeled stress periods.*

This is the core comparison for the research question: do our signals warn **earlier** than the standard indicators (VIX and the St. Louis Fed Financial Stress Index, STLFSI4)?

**Signals compared**
| Signal | Rule |
|---|---|
| Network anomaly (combined) | notebook 13 alert: z-score or stress-direction Isolation Forest |
| Network z-score only | any network metric z >= 2 |
| Model alert | "+ z-scores + IF" model from notebook 14 (top 10% probability) |
| Baseline model alert | VIX + past volatility model from notebook 14 |
| VIX > 25 | common rule of thumb for an elevated VIX |
| VIX z-score | VIX at least 2 standard deviations above its previous year |
| STLFSI4 > 0 / > 1 | above-average stress / clearly elevated; **lagged 5 trading days** for publication |

**Scoring** (`src/alerts.py`): for each episode, the first alert in the 60 trading days before its start (not reaching back into the previous episode) gives the lead time. First alerts in the 10 days after the start count as late. Alert days are grouped into events; an event that is not followed by an episode start within 60 days, and does not start inside an episode, is a false alarm. Both start dates are reported (`start` and `start_trailing`).

**STLFSI4 lag:** the panel carries STLFSI4 on its observation date. As far as I know it is published weekly, several days after the week it covers, so it is shifted 5 trading days here. This lag is an assumption to verify on FRED before the final report.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works from the repo root or from notebooks/. RGBFS_DATA_ROOT is only for test runs on fake data.
CODE_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "anomaly.py").exists())
DATA_ROOT = Path(os.environ.get("RGBFS_DATA_ROOT", CODE_ROOT))
sys.path.insert(0, str(CODE_ROOT))

from src import anomaly as an
from src import alerts as al

PROC = DATA_ROOT / "data" / "processed"
FEATURES = PROC / "features"
EPISODES_CSV = DATA_ROOT / "docs" / "stress_periods.csv"
OUT = DATA_ROOT / "notebooks" / "outputs" / "rishi_week6"
OUT.mkdir(parents=True, exist_ok=True)
print("Code:", CODE_ROOT)
print("Data:", DATA_ROOT)

In [ ]:
def shade_episodes(ax, episodes, color="tab:red", alpha=0.12):
    for _, ep in episodes.iterrows():
        ax.axvspan(ep.start, ep.end, color=color, alpha=alpha, lw=0)

## 1. Build every signal on the trading-day calendar

In [ ]:
panel = pd.read_parquet(PROC / "daily_panel.parquet").sort_index()
panel.index = pd.to_datetime(panel.index)
cal = panel.index
episodes = al.load_episodes(EPISODES_CSV)
anom = pd.read_parquet(FEATURES / "anomaly_network_daily.parquet").reindex(cal)

vix = panel["VIXCLS"].ffill(limit=3)
vix_z = an.trailing_zscores(vix.to_frame("vix"), lookback=252, min_periods=126)["vix"]
stlfsi = panel["STLFSI4"].shift(5)
start_common = anom.alert.first_valid_index()   # compare all signals over the same period

def from_common(s):
    s = s.astype(float).where(s.notna())
    return s.where(s.index >= start_common)

signals = {
    "Network anomaly (combined)": anom.alert,
    "Network z-score only": anom.z_flag,
    "VIX > 25": (vix > 25).astype(float).where(vix.notna()),
    "VIX z >= 2": (vix_z >= 2).astype(float).where(vix_z.notna()),
    "STLFSI4 > 0 (lagged)": (stlfsi > 0).astype(float).where(stlfsi.notna()),
    "STLFSI4 > 1 (lagged)": (stlfsi > 1).astype(float).where(stlfsi.notna()),
}
model_path = DATA_ROOT / "notebooks" / "outputs" / "rishi_week5" / "model_alerts_daily.csv"
if model_path.exists():
    m = pd.read_csv(model_path, index_col="date", parse_dates=True).reindex(cal)
    test = (cal >= m.index[m["baseline|alert"].notna()].min()) & (cal <= m.index[m["baseline|alert"].notna()].max())
    for label, col in [("Model alert (+ z + IF)", "+ z-scores + IF|alert"), ("Baseline model alert (VIX + vol)", "baseline|alert")]:
        signals[label] = m[col].where(~(test & m[col].isna()), 0.0).where(test)
else:
    print("Run notebook 14 first to include the model alerts")
signals = {k: from_common(v) for k, v in signals.items()}
print("Common start:", start_common.date())

## 2. Summary for each signal

In [ ]:
summary = pd.DataFrame([al.summarize_signal(name, s, episodes, start_col=col)
                        for col in ["start", "start_trailing"] for name, s in signals.items()])
cols = ["signal", "start_col", "episodes_scored", "warned_early", "late", "missed", "median_lead_days",
        "false_alarms_per_year", "day_precision", "day_recall", "day_false_alarm_rate"]
summary[cols].round(3)

## 3. Lead time by episode

Positive = warned that many trading days before the start. Negative = late. Blank = missed or no data for that signal yet. `*` = the signal was already on when the 60-day window opened, so the lead is capped by the window rather than a clean early warning.

In [ ]:
def lead_table(start_col):
    cols = {}
    for name, s in signals.items():
        lt = al.lead_times(s, episodes, start_col=start_col)
        cols[name] = [("" if not r.has_data else ("missed" if pd.isna(r.lead_days) else f"{r.lead_days:.0f}{'*' if r.already_on else ''}"))
                      for r in lt.itertuples()]
    t = pd.DataFrame(cols)
    t.insert(0, "start", al.lead_times(next(iter(signals.values())), episodes, start_col=start_col)["start"].dt.date)
    t.insert(0, "episode", episodes["name"])
    return t

leads_start = lead_table("start")
leads_trailing = lead_table("start_trailing")
leads_start

In [ ]:
leads_trailing

## 4. Network anomaly vs. VIX, head to head

For each episode both signals scored: how many trading days earlier (positive) or later (negative) the network anomaly fired than VIX z >= 2 and than VIX > 25. A missed signal is treated as firing at the end of the window, so a miss always loses.

In [ ]:
def first_day_lead(s, col):
    lt = al.lead_times(s, episodes, start_col=col)
    return lt.lead_days.where(lt.has_data).fillna(-al.LATE - 1).where(lt.has_data)

h2h = pd.DataFrame({"episode": episodes["name"]})
for col in ["start", "start_trailing"]:
    net = first_day_lead(signals["Network anomaly (combined)"], col)
    for v in ["VIX z >= 2", "VIX > 25"]:
        h2h[f"vs {v} ({col})"] = net - first_day_lead(signals[v], col)
h2h

## 5. Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 0.6 * len(signals) + 1.5))
for i, (name, s) in enumerate(signals.items()):
    on = s.index[s.eq(1)]
    ax.scatter(on, np.full(len(on), i), marker="|", s=60, lw=1)
ax.set_yticks(range(len(signals)), list(signals))
shade_episodes(ax, episodes, alpha=0.2)
ax.set_title("Alert days by signal (red = labeled stress episodes)")
plt.tight_layout(); plt.savefig(OUT / "alert_timeline.png", dpi=120); plt.show()

## 6. Save outputs

The dashboard's data/alerts panel reads these files.

In [ ]:
summary.to_csv(OUT / "signal_summary.csv", index=False)
leads_start.to_csv(OUT / "lead_times_by_episode_start.csv", index=False)
leads_trailing.to_csv(OUT / "lead_times_by_episode_start_trailing.csv", index=False)
h2h.to_csv(OUT / "network_vs_vix.csv", index=False)
pd.DataFrame(signals).rename_axis("date").to_csv(OUT / "signals_daily.csv")
events = pd.concat([al.score_events(s, episodes).assign(signal=name) for name, s in signals.items()], ignore_index=True)
events.to_csv(OUT / "alert_events.csv", index=False)
print("Saved to", OUT)

## 7. Findings

*To fill in after the real-data run:*
* Which signal warned the most episodes early, and with what median lead?
* Did the network anomaly fire before VIX (Section 4), and in which episodes?
* What did the early warnings cost in false alarms per year?
* Do the answers change between `start` and `start_trailing`?
* Read lead times together with false alarms per year. A signal that fires often will "warn" many episodes by chance; its first alert in the 60-day window may just be noise.
* Earlier EDAs found mixed evidence (average correlation led VIX in 2011 but not in 2020 or 2022). Say whether this full-history test agrees.